# Entrainement et versioning du modele

Ce notebook reprend le pipeline actuel du projet `ml_conso` : chargement des donnees, creation de la cible `evo_conso`, selection des features, entrainement, evaluation et sauvegarde du modele actif.

## 1. Initialiser les chemins du projet

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"
METRICS_DIR = ARTIFACTS_DIR / "metrics"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Projet    :", PROJECT_ROOT)
print("Src       :", SRC_PATH)
print("Artifacts :", ARTIFACTS_DIR)

## 2. Importer les fonctions du projet

In [ ]:
import json
import joblib
import mlflow
import mlflow.sklearn
import pandas as pd

from src.ml_conso.data import load_data
from src.ml_conso.features import FEATURES, TARGET, select_features, split_data
from src.ml_conso.pipeline import (
    activate_model_version,
    build_pipeline,
    export_model_contract,
    save_model_version,
)
from src.ml_conso.evaluate import evaluate_model
from src.ml_conso.train import log_experiment

## 3. Charger les donnees

In [ ]:
df_raw = load_data()

print("Shape brute :", df_raw.shape)
df_raw.head()

## 4. Verifier les colonnes utiles

In [ ]:
required_columns = ["Date", "Conso_MWH"] + FEATURES
missing_columns = [column for column in required_columns if column not in df_raw.columns]

print("Target :", TARGET)
print("Features :", FEATURES)
print("Colonnes manquantes :", missing_columns)

assert not missing_columns, f"Colonnes manquantes : {missing_columns}"

## 5. Creer la cible et selectionner les features

In [ ]:
df_model = select_features(df_raw.copy())

print("Shape modele :", df_model.shape)
print("Colonnes modele :", df_model.columns.tolist())
df_model.head()

## 6. Controler la qualite des donnees modele

In [ ]:
missing_summary = df_model.isna().sum().sort_values(ascending=False)
target_summary = df_model[TARGET].describe()

display(missing_summary.to_frame("missing_values"))
display(target_summary.to_frame("evo_conso"))

## 7. Split train / test

In [ ]:
X_train, X_test, y_train, y_test = split_data(df_model)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

assert TARGET not in X_train.columns
assert y_train.name == TARGET

## 8. Construire le pipeline

In [ ]:
pipeline = build_pipeline()

print(pipeline)
print("Etapes :", list(pipeline.named_steps))

## 9. Entrainer le modele

In [ ]:
pipeline.fit(X_train, y_train)

print("Modele entraine")

## 10. Evaluer le modele

In [ ]:
metrics = evaluate_model(pipeline, X_test, y_test)

metrics_df = pd.DataFrame([metrics])
display(metrics_df)
metrics

## 11. Sauvegarder et activer le modele courant

In [ ]:
mlflow.set_experiment("electricity_forecasting")

with mlflow.start_run():
    log_experiment(metrics)

    saved_paths = save_model_version(pipeline, metrics, "current", ARTIFACTS_DIR)
    latest_paths = activate_model_version("current", ARTIFACTS_DIR)

    mlflow.sklearn.log_model(sk_model=pipeline, artifact_path="model")

    feature_columns_path = ARTIFACTS_DIR / "feature_columns.pkl"
    joblib.dump(X_train.columns.tolist(), feature_columns_path)

    contract_path = export_model_contract(X_train.columns.tolist(), ARTIFACTS_DIR)

print("Version current :", saved_paths)
print("Version latest  :", latest_paths)
print("Features       :", feature_columns_path)
print("Contract       :", contract_path)

## 12. Verifier les artefacts generes

In [ ]:
expected_artifacts = [
    MODELS_DIR / "conso_model_current.joblib",
    MODELS_DIR / "conso_model_latest.joblib",
    METRICS_DIR / "conso_metrics_current.json",
    METRICS_DIR / "conso_metrics_latest.json",
    ARTIFACTS_DIR / "conso_model_contract.json",
    ARTIFACTS_DIR / "conso_feature_columns.pkl",
]

artifact_status = pd.DataFrame(
    {
        "path": [str(path.relative_to(PROJECT_ROOT)) for path in expected_artifacts],
        "exists": [path.exists() for path in expected_artifacts],
    }
)

display(artifact_status)
assert artifact_status["exists"].all()

## 13. Lire le contrat du modele

In [ ]:
with (ARTIFACTS_DIR / "model_contract.json").open(encoding="utf-8") as file:
    model_contract = json.load(file)

model_contract

## 14. Pistes d'amelioration du modele

1. Reintegrer des variables temporelles : jour de semaine, semaine de l'annee, mois, saison.
2. Reintegrer `CODE_DEPARTEMENT` pour capter les differences geographiques.
3. Ajouter davantage de variables meteo : vent, precipitation, neige, moyenne temperature, amplitude thermique.
4. Comparer le pipeline actuel a l'ancien modele enrichi `v1/v2`, qui utilisait plus de features.
5. Tester plusieurs modeles : ExtraTreesRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor.
6. Ajouter une validation croisee ou un split temporel si l'objectif est de predire des periodes futures.
7. Suivre un seuil anti-regression dans les tests, par exemple conserver `r2 > 0.80` sur l'artefact latest.